In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import logging
import os
import re
from collections import Counter

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser

logger = logging.getLogger('notebook')

plt.style.use('default')
plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')


In [ ]:
METRIC = "correct"

In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()

p_standard_pilot = pp / "mini_20x50x4__14_11/final"
p_standard = pp / "default_full__06_02/final"

p_code = pp / 'code_full__09_02/corrected'
p_formal = pp / 'formalised_full__16_04/final'
p_nonformal = pp / 'nonformalised_full__20_04/final'

In [ ]:
# mres_standard_pilot = MultiVariantMultiModelResultsAnalyser(p_standard_pilot)

mres_standard = MultiVariantMultiModelResultsAnalyser(p_standard)

# mres_code = MultiVariantMultiModelResultsAnalyser(p_code)
# mres_formal = MultiVariantMultiModelResultsAnalyser(p_formal)
# mres_nonformal = MultiVariantMultiModelResultsAnalyser(p_nonformal)


In [ ]:
OUTPUTS = Path("./outputs").resolve()
os.makedirs(OUTPUTS, exist_ok=True)
OUTPUTS_FOLDER = str(OUTPUTS) + "/"

In [ ]:
mres_standard.models

In [ ]:
mres_standard.variants['main'].full_data

In [ ]:


first_model = mres_standard.models[0]
number_pattern = re.compile(r'\d+(?:\.\d+)?')
integer_bin_edges = [0, 1, 2, 3, 4, 5, 10, 20, 50, 100, 1000, float('inf')]
raw_integer_bin_labels = [f'[{integer_bin_edges[i]}, {integer_bin_edges[i+1]})' for i in range(len(integer_bin_edges) - 1)]
bin_labels = ['fractions']
for start, end in zip(integer_bin_edges[:-1], integer_bin_edges[1:]):
    if np.isinf(end):
        label = f"{start}+"
    elif (end - start) == 1:
        label = f"{start}"
    else:
	    label = f'{start}-{end}'
    bin_labels.append(label)

binned_counts_dict = {}
raw_counts_dict = {}
for variant_name, analyser in mres_standard.variants.items():
    variant_df = analyser.full_data
    model_df = variant_df.loc[variant_df['model'] == first_model, ['question']].copy()
    extracted_numbers = (
        model_df['question']
        .str.findall(number_pattern)
        .explode()
        .dropna()
        .astype(float)
    )
    raw_counts_dict[variant_name] = Counter(extracted_numbers)
    integer_mask = np.isclose(extracted_numbers, np.round(extracted_numbers))
    integer_numbers = extracted_numbers[integer_mask]
    fraction_count = int((~integer_mask).sum())
    integer_binned = pd.cut(
        integer_numbers,
        bins=integer_bin_edges,
        labels=raw_integer_bin_labels,
        right=False,
        include_lowest=True,
    )
    integer_counts = integer_binned.value_counts().reindex(raw_integer_bin_labels, fill_value=0)
    binned_counts = pd.Series(
        [fraction_count] + list(integer_counts.iloc[:5].values) + list(integer_counts.iloc[5:].values),
        index=bin_labels,
        dtype=int,
    )
    binned_counts_dict[variant_name] = binned_counts

raw_counts_df = pd.DataFrame(raw_counts_dict).fillna(0).astype(int).sort_index()
binned_counts_df = pd.DataFrame(binned_counts_dict).fillna(0).astype(int)
binned_counts_df

In [ ]:
plot_bin_positions = np.arange(len(bin_labels))
plot_bin_centers = plot_bin_positions + 0.5
variants = list(binned_counts_df.columns)
n_variants = len(variants)
variant_colors = {
    'GSM8K': 'mediumslateblue',
    'main': 'darksalmon',
}
fig, (ax_count, ax_cum) = plt.subplots(2, 1, figsize=(10, 7))
total_bar_width = 0.8
bar_width = total_bar_width / max(n_variants, 1)
bar_offset = (1.0 - total_bar_width) / 2.0
for idx, variant_name in enumerate(variants):
    percentages = binned_counts_df[variant_name] / binned_counts_df[variant_name].sum() * 100
    bar_positions = plot_bin_positions + bar_offset + idx * bar_width
    color = variant_colors.get(variant_name, None)
    ax_count.bar(
        bar_positions,
        percentages,
        width=bar_width,
        align='edge',
        alpha=0.8,
        edgecolor='white',
        linewidth=0.8,
        label=variant_name,
        color=color,
    )

    raw_counts = raw_counts_df[variant_name]
    raw_counts = raw_counts[raw_counts > 0]
    cum_perc = raw_counts.cumsum() / raw_counts.sum() * 100
    ax_cum.plot(cum_perc.index, cum_perc,
                marker='.', ms=4, lw=1.0, label=variant_name, color=color)

for ax in (ax_count, ax_cum):
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
    ax.legend(title='Dataset variant')

ax_count.set_xticks(plot_bin_centers)
ax_count.set_xticklabels(bin_labels)
ax_count.set_xlabel('Extracted number buckets')
ax_count.set_ylabel('Percent of variant total')

ax_cum.set_xlabel('Extracted numbers')
ax_cum.set_ylabel('Cumulative percent')
ax_cum.set_xlim(-5, 105)

fig.tight_layout()
